# 11. Additional experiments

This notebook explores several targeted extensions aimed at understanding the main limitations of the predictive pipeline.

The experiments focus on four questions:
1. Does stronger filtering improve label quality and predictive performance?
2. Does dimensionality reduction help recover signal from the embedding space?
3. Do SigLIP and DINOv2 contain complementary information?
4. Can Gradient Boosting outperform Random Forest on these embedding-based tasks?

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.decomposition import PCA

In [2]:
def evaluate_regression(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, rmse

def run_models(X, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )

    baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
    baseline_r2, baseline_rmse = evaluate_regression(y_test, baseline_pred)

    rf = RandomForestRegressor(n_estimators=100, random_state=random_state, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_r2, rf_rmse = evaluate_regression(y_test, rf_pred)

    gb = GradientBoostingRegressor(n_estimators=200, random_state=random_state)
    gb.fit(X_train, y_train)
    gb_pred = gb.predict(X_test)
    gb_r2, gb_rmse = evaluate_regression(y_test, gb_pred)

    return pd.DataFrame([
        ["Baseline", baseline_r2, baseline_rmse],
        ["Random Forest", rf_r2, rf_rmse],
        ["Gradient Boosting", gb_r2, gb_rmse]
    ], columns=["Model", "R²", "RMSE"])

## 1. Stronger filtering for warmth

This experiment tests whether increasing the minimum number of matched warmth descriptors improves predictive performance by reducing label noise.

In [3]:
siglip_strict = pd.read_csv("../Embeddings/siglip/final_dataset_strict_with_siglip.csv")

exp1_df = siglip_strict[siglip_strict["n_dirmean_Warmth"] >= 5].copy()

emb_cols = [c for c in exp1_df.columns if c.startswith("emb_")]
X_exp1 = exp1_df[emb_cols].values
y_exp1 = exp1_df["dirmean_Warmth"].values

print("Rows after stronger filtering:", len(exp1_df))
results_exp1 = run_models(X_exp1, y_exp1)
results_exp1

Rows after stronger filtering: 2040


,Model,R²,RMSE
0,Baseline,-0.003510,0.352291
1,Random Forest,0.171461,0.320109
2,Gradient Boosting,0.153389,0.323581


## Interpretation: Stronger filtering

Applying a stricter filtering criterion based on the number of warmth-related descriptors leads to a substantial improvement in predictive performance.

Compared to previous experiments, the Random Forest model shows a clear increase in R², reaching approximately 0.17. This represents a significant improvement over earlier results obtained on the full dataset.

This result suggests that label quality plays a critical role in this task. By restricting the dataset to observations with more reliable warmth annotations, the model is able to recover a stronger relationship between visual embeddings and the target variable.

Importantly, this indicates that the limited performance observed in previous experiments was not solely due to model limitations, but largely driven by noise in the target labels. When this noise is reduced, the predictive signal becomes much more apparent.

Overall, this experiment supports the idea that visual embeddings do contain meaningful information related to perceived warmth, but that this signal is highly sensitive to the quality of the annotations used.

Given this strong sensitivity to data quality, the next experiment explores whether reducing noise in the feature space through dimensionality reduction (PCA) can further improve performance.

## 2. Dimensionality reduction (PCA)

This experiment evaluates whether reducing the dimensionality of the embedding space improves predictive performance.

High-dimensional embeddings may contain noise or redundant information. PCA is applied to project the features into a lower-dimensional space while preserving the most important variance.

The goal is to determine whether the predictive signal is concentrated in a smaller set of components.

In [4]:
from sklearn.decomposition import PCA

# Use the SAME filtered dataset from Experiment 1
pca_100 = PCA(n_components=100)

X_pca_100 = pca_100.fit_transform(X_exp1)

print("Original shape:", X_exp1.shape)
print("PCA shape:", X_pca_100.shape)

results_pca_100 = run_models(X_pca_100, y_exp1)
results_pca_100

Original shape: (2040, 768)
PCA shape: (2040, 100)


,Model,R²,RMSE
0,Baseline,-0.003510,0.352291
1,Random Forest,0.176605,0.319113
2,Gradient Boosting,0.131302,0.327775


In [5]:
pca_50 = PCA(n_components=50)

X_pca_50 = pca_50.fit_transform(X_exp1)

print("PCA 50 shape:", X_pca_50.shape)

results_pca_50 = run_models(X_pca_50, y_exp1)
results_pca_50

PCA 50 shape: (2040, 50)


,Model,R²,RMSE
0,Baseline,-0.003510,0.352291
1,Random Forest,0.181226,0.318217
2,Gradient Boosting,0.139355,0.326252


## Interpretation: PCA

Applying PCA to reduce the dimensionality of the embedding space leads to a slight but consistent improvement in performance.

With 100 components, Random Forest achieves an R² of approximately 0.18, slightly improving over the non-reduced case. Reducing further to 50 components maintains this improvement and even yields the best result among all tested configurations.

These findings suggest that the predictive signal is not distributed across all embedding dimensions, but instead concentrated in a smaller subset of components. By removing noisy or redundant features, PCA helps the model focus on the most informative aspects of the representation.

Importantly, the improvement is modest rather than dramatic. This indicates that while dimensionality reduction helps reduce noise in the feature space, it does not fundamentally change the strength of the relationship between visual embeddings and the target variable.

Overall, this experiment shows that both label quality and feature noise play a role in limiting performance. While stronger filtering had a larger impact, PCA provides an additional, complementary improvement.

## Interpretation: PCA

Applying PCA to reduce the dimensionality of the embedding space leads to a slight but consistent improvement in performance.

With 100 components, Random Forest achieves an R² of approximately 0.18, slightly improving over the non-reduced case. Reducing further to 50 components maintains this improvement and even yields the best result among all tested configurations.

These findings suggest that the predictive signal is not distributed across all embedding dimensions, but instead concentrated in a smaller subset of components. By removing noisy or redundant features, PCA helps the model focus on the most informative aspects of the representation.

Importantly, the improvement is modest rather than dramatic. This indicates that while dimensionality reduction helps reduce noise in the feature space, it does not fundamentally change the strength of the relationship between visual embeddings and the target variable.

Overall, this experiment shows that both label quality and feature noise play a role in limiting performance. While stronger filtering had a larger impact, PCA provides an additional, complementary improvement.

In [6]:
siglip_df = pd.read_csv("../Embeddings/siglip/final_dataset_strict_with_siglip.csv")
dino_df = pd.read_csv("../Embeddings/dinov2/final_dataset_strict_with_dinov2.csv")

print("SigLIP shape:", siglip_df.shape)
print("DINOv2 shape:", dino_df.shape)

SigLIP shape: (3060, 789)
DINOv2 shape: (3060, 789)


In [7]:
merged_df = siglip_df.merge(dino_df, on="cat_no", suffixes=("_siglip", "_dino"))

print("Merged shape:", merged_df.shape)

Merged shape: (3060, 1577)


In [8]:
merged_df = merged_df[merged_df["n_dirmean_Warmth_siglip"] >= 5].copy()

print("Rows after filtering:", len(merged_df))

Rows after filtering: 2040


In [9]:
siglip_cols = [c for c in merged_df.columns if c.startswith("emb_") and "_siglip" in c]
dino_cols = [c for c in merged_df.columns if c.startswith("emb_") and "_dino" in c]

X_sig = merged_df[siglip_cols].values
X_dino = merged_df[dino_cols].values

X_combined = np.concatenate([X_sig, X_dino], axis=1)

y_combined = merged_df["dirmean_Warmth_siglip"].values

print("SigLIP shape:", X_sig.shape)
print("DINO shape:", X_dino.shape)
print("Combined shape:", X_combined.shape)

SigLIP shape: (2040, 768)
DINO shape: (2040, 768)
Combined shape: (2040, 1536)


In [10]:
results_combined = run_models(X_combined, y_combined)
results_combined

,Model,R²,RMSE
0,Baseline,-0.003510,0.352291
1,Random Forest,0.159415,0.322427
2,Gradient Boosting,0.152113,0.323825


## Interpretation: Combined embeddings (SigLIP + DINOv2)

Combining embeddings from SigLIP and DINOv2 does not lead to an improvement in predictive performance.

After concatenating both representations, the Random Forest model achieves an R² of approximately 0.16, which is slightly lower than the best result obtained using SigLIP embeddings alone (~0.18). A similar pattern is observed for Gradient Boosting.

This suggests that the two embedding models do not provide strongly complementary information for this task. Instead, the additional features introduced by combining both representations appear to introduce noise or redundancy, which slightly degrades performance.

These findings indicate that the predictive signal related to perceived warmth is already largely captured by a single embedding model, and that combining multiple visual encoders does not necessarily enhance performance.

Overall, this experiment reinforces the idea that the main limitation lies not in the choice or diversity of visual encoders, but in the strength and clarity of the underlying signal itself.